#**Retrieval-Augmented Generation (RAG) System with ChromaDB + Gemini**
## **Project Overview**

This notebook demonstrates the complete implementation of a Retrieval-Augmented Generation (RAG) pipeline using:

 * PyMuPDF for PDF extraction
 * Sentence Transformers for embeddings
 * ChromaDB as a vector database
 * BM25 for keyword retrieval
 * Cross-Encoder for reranking
 * Open router API for answer generation
 * Streamlit & Gradio for deployment interfaces

The system processes PDF documents, converts them into semantic chunks, stores embeddings in a vector database, retrieves the most relevant chunks for a query, and finally generates accurate answers using an LLM.

# **RAG Pipeline Architecture**

```text
PDF
 ↓
Cleaning
 ↓
Recursive Chunking
 ↓
Embeddings (E5-Large)
 ↓
ChromaDB
 ↓
Hybrid Search (BM25 + Vector)
 ↓
Cross-Encoder Reranking
 ↓
Openrouter Generation

# **1. Install Required Libraries**

In [ ]:
# PDF Processing
!pip install pymupdf nltk tqdm

# Embeddings & Vector Database
!pip install sentence-transformers chromadb

# Hybrid Search
!pip install rank_bm25

# Gemini API
!pip install -U -q google-generativeai

# Web Interfaces
!pip install -q gradio

# Streamlit Tunnel
!npm install localtunnel

⠙⠹⠸⠼⠴⠦
up to date, audited 23 packages in 849ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠧

# **2. Import Libraries**

In [ ]:
import fitz  # PyMuPDF
import re
import nltk
import json
import numpy as np
import chromadb
import torch
import time

from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google.api_core import exceptions
import google.generativeai as genai

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# **3. Upload PDF Files**

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_files = list(uploaded.keys())

print("Uploaded files:", pdf_files)

Saving TM Lecture 8.pdf to TM Lecture 8.pdf
Uploaded files: ['TM Lecture 8.pdf']


# **4. Extract Text from PDFs**

- This function extracts text page by page from uploaded PDF files.

In [ ]:
import fitz

def extract_pages(file_path):
    doc = fitz.open(file_path)

    pages = []
    for i, page in enumerate(doc):
        text = page.get_text()
        pages.append({
            "page_num": i + 1,
            "text": text
        })

    return pages

In [ ]:
all_extracted_pages = []
for file_name in pdf_files:
    all_extracted_pages.extend(extract_pages(file_name))

pages = all_extracted_pages
print("Total Pages:", len(pages))

Total Pages: 19


# **5. Remove Headers & Footers**

Many PDFs contain repeated headers and footers.
This step automatically detects and removes them.

## **Detect Repeated Lines**

In [ ]:
def detect_repeated_lines(pages, threshold=0.6):
    lines_all = []

    for p in pages:
        lines = p["text"].split("\n")
        # Candidate header/footer lines
        candidate_lines = lines[:3] + lines[-3:]
        lines_all.extend(candidate_lines)

    counter = Counter(lines_all)
    total_pages = len(pages)

    repeated = set()
    for line, count in counter.items():
        if count / total_pages >= threshold and len(line.strip()) > 3:
            repeated.add(line.strip())

    return repeated

## **Remove Detected Lines**

In [ ]:
def remove_headers_footers(pages):
    repeated_lines = detect_repeated_lines(pages)

    cleaned_pages = []
    for p in pages:
        lines = p["text"].split("\n")
        new_lines = []

        for line in lines:
            if line.strip() not in repeated_lines:
                new_lines.append(line)

        cleaned_pages.append({
            "page_num": p["page_num"],
            "text": "\n".join(new_lines)
        })

    return cleaned_pages

## **6. Text Cleaning**

This function cleans noisy text by removing:

 * URLs
 * Emails
 * Page numbers
 * Excessive symbols
 * Repeated characters
 * Extra spaces

In [ ]:
def clean_text(text):

    # URLs & emails
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)

    # Page numbers
    text = re.sub(r'page\s*\d+', ' ', text, flags=re.IGNORECASE)

    # Remove PDF artifacts
    text = re.sub(r'\[|\]', ' ', text)

    # Fix comma spacing
    text = re.sub(r'\s*,\s*', ', ', text)

    # Fix broken tokens (and, I → and I)
    text = re.sub(r'(\w),(\w)', r'\1 \2', text)

    # Remove weird symbols
    text = re.sub(r'[^\x00-\x7F\u0600-\u06FF]+', ' ', text)

    # Fix repeated punctuation
    text = re.sub(r'([.,!?]){2,}', r'\1', text)

    # Remove repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

## **7. Text Chunking**



Large documents are divided into smaller chunks for better retrieval performance.

In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def chunk_text(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    chunks = splitter.split_text(text)

    return chunks

## **8. Process PDFs Pipeline**



This function combines:

 * Extraction
 * Cleaning
 * Chunking

into a single pipeline.

In [ ]:
def process_pdfs(pdf_files):

    all_chunks = []

    for file in pdf_files:

        print(f"\nProcessing: {file}")

        pages = extract_pages(file)
        pages = remove_headers_footers(pages)

        for p in pages:

            clean = clean_text(p["text"])

            chunks = chunk_text(clean)

            print(f"\nPage {p['page_num']}")
            print(f"Characters: {len(clean)}")
            print(f"Words: {len(clean.split())}")
            print(f"Chunks: {len(chunks)}")

            for i, chunk in enumerate(chunks):

                # Skip very small chunks
                if len(chunk.split()) < 15:
                    continue

                all_chunks.append({
                    "text": chunk,
                    "source": file,
                    "page": p["page_num"],
                    "chunk_id": f"{file}_p{p['page_num']}_c{i}",
                    "word_count": len(chunk.split())
                })

    return all_chunks

## **9. Generate Chunks**



In [ ]:
all_chunks = process_pdfs(pdf_files)

print("Total chunks:", len(all_chunks))


Processing: TM Lecture 8.pdf

Page 1
Characters: 264
Words: 38
Chunks: 1

Page 2
Characters: 338
Words: 48
Chunks: 1

Page 3
Characters: 336
Words: 47
Chunks: 1

Page 4
Characters: 365
Words: 53
Chunks: 1

Page 5
Characters: 348
Words: 53
Chunks: 1

Page 6
Characters: 332
Words: 42
Chunks: 1

Page 7
Characters: 288
Words: 40
Chunks: 1

Page 8
Characters: 176
Words: 33
Chunks: 1

Page 9
Characters: 266
Words: 42
Chunks: 1

Page 10
Characters: 317
Words: 49
Chunks: 1

Page 11
Characters: 127
Words: 29
Chunks: 1

Page 12
Characters: 241
Words: 44
Chunks: 1

Page 13
Characters: 281
Words: 42
Chunks: 1

Page 14
Characters: 204
Words: 34
Chunks: 1

Page 15
Characters: 267
Words: 46
Chunks: 1

Page 16
Characters: 255
Words: 37
Chunks: 1

Page 17
Characters: 206
Words: 32
Chunks: 1

Page 18
Characters: 162
Words: 29
Chunks: 1

Page 19
Characters: 272
Words: 55
Chunks: 1
Total chunks: 19


# **10. Save Chunks as JSON**

In [ ]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Saved to chunks.json")

Saved to chunks.json


In [ ]:
for i in range(3):
    print(all_chunks[i])

{'text': 'Language Modeling and N-grams Introduction to Language Modeling Language modeling focuses on predicting the probability of sequences of words. It enables computers to understand and generate human language. Example: completing the sentence "The cat sat on the __."', 'source': 'TM Lecture 8.pdf', 'page': 1, 'chunk_id': 'TM Lecture 8.pdf_p1_c0', 'word_count': 38}
{'text': 'Why Language Modeling Matters Powers everyday applications such as smartphones, email, and social media. Acts as a bridge between humans and machines. Forms the foundation of AI assistants like Siri, Alexa, and ChatGPT. Real-World Applications Predictive text and autocorrect in messaging apps. Search engines that understand user intent.', 'source': 'TM Lecture 8.pdf', 'page': 2, 'chunk_id': 'TM Lecture 8.pdf_p2_c0', 'word_count': 48}
{'text': 'Machine translation and speech recognition. Virtual assistants that respond to voice commands. How Language Models Work Learn patterns from large text datasets. Store w

# **11. Embedding Generation**



We use:

* Model: ```text intfloat/multilingual-e5-large```
* Vector DB: **ChromaDB**

In [ ]:
# Check GPU
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


In [ ]:
#Load chunks
with open("chunks.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f" Loaded {len(all_chunks)} chunks")

 Loaded 19 chunks


## **Load Embedding Model**

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-large"
print(f" Loading model: {MODEL_NAME}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

config = {
    "prefix_query": "query: "
}

print(f" Model loaded on: {device}")

 Loading model: intfloat/multilingual-e5-large


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

 Model loaded on: cuda


## **Generate Embeddings**

In [ ]:
texts = [
    "passage: " + chunk["text"]
    for chunk in all_chunks
]

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device
)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (19, 1024)


# **12. Setup ChromaDB**

In [ ]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

try:
    chroma_client.delete_collection(
        "rag_collection"
    )
except:
    pass

collection = chroma_client.create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created")

ChromaDB collection created


# **13. Add Data to ChromaDB**

In [ ]:
ids = [f"id_{i}" for i in range(len(all_chunks))]

metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"]
    }
    for chunk in all_chunks
]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=[chunk["text"] for chunk in all_chunks],
    metadatas=metadatas
)

print(f"Added {collection.count()} chunks")

Added 19 chunks


# **14. Vector Search**

This function retrieves the most relevant chunks using semantic similarity.

In [ ]:
def search(query, top_k=5):

    query_vec = model.encode(
        ["query: " + query],
        normalize_embeddings=True,
        device=device
    ).tolist()

    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    print(f"\nQuery: {query}\n")

    for rank, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):

        print(
            f"[{rank+1}] "
            f"Score: {1-dist:.4f} | "
            f"Page: {meta['page']} | "
            f"Source: {meta['source']}"
        )

        print(f"{doc[:120]}...\n")

In [ ]:
search("What is sentiment analysis?", top_k=5)


Query: What is sentiment analysis?

[1] Score: 0.8425 | Page: 4 | Source: TM Section 1.pdf
Case Study 1: Sentiment Analysis of Two Reviews Step 2: Define Sentiment Dictionary A Sentiment Dictionary, or Lexicon, ...

[2] Score: 0.8360 | Page: 5 | Source: TM Section 1.pdf
Case Study 1: Sentiment Analysis of Two Reviews Step 3: Compute Sentiment Score This is the quantitative phase where ind...

[3] Score: 0.8284 | Page: 2 | Source: TM Section 1.pdf
Case Study 1: Sentiment Analysis of Two Reviews We have two customer reviews: Review 1: "The product is amazing! It work...

[4] Score: 0.8268 | Page: 5 | Source: TM Section 1.pdf
. Total Sentiment = Sum(Polarity of each word) Review 1: 1 (amaze) + 1 (perfectly) + 1 (love) = +3 Review 2: -1 (terribl...

[5] Score: 0.8252 | Page: 3 | Source: TM Section 1.pdf
Case Study 1: Sentiment Analysis of Two Reviews Step 1: Preprocess & Clean Text 1. Tokenization: We break each sentence ...



# **15. Hybrid Search (Semantic + BM25)**

Hybrid Search combines:

 * Semantic Search
 * Keyword Search

for improved retrieval quality.

In [ ]:
tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready")

BM25 index ready


In [ ]:
def hybrid_search(query, top_k=10, alpha=0.6):
    global model, collection, config

    #  Vector Search scores
    query_vec = model.encode(
        [config["prefix_query"] + query],
        normalize_embeddings=True,
        device=device
    ).tolist()

    vec_results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # dict: text → vector_score
    vec_scores = {}
    vec_meta   = {}
    for doc, meta, dist in zip(
        vec_results["documents"][0],
        vec_results["metadatas"][0],
        vec_results["distances"][0]
    ):
        vec_scores[doc] = round(1 - dist, 4)
        vec_meta[doc]   = meta

    #  BM25 scores
    tokenized_query = query.lower().split()
    bm25_raw        = bm25.get_scores(tokenized_query)

    # Normalize BM25 scores
    bm25_max = bm25_raw.max()
    if bm25_max > 0:
        bm25_norm = bm25_raw / bm25_max
    else:
        bm25_norm = bm25_raw

    #vector_score+BM25 score
    combined = {}

    #  vector scores
    for doc, vscore in vec_scores.items():
        combined[doc] = {"vector": vscore, "bm25": 0.0, "meta": vec_meta[doc]}

    #  BM25 scores
    for i, chunk in enumerate(all_chunks):
        text = chunk["text"]
        if text in combined:
            combined[text]["bm25"] = round(float(bm25_norm[i]), 4)

    #  4. hybrid score
    output = []
    for doc, scores in combined.items():
        hybrid_score = alpha * scores["vector"] + (1 - alpha) * scores["bm25"]
        output.append({
            "text"        : doc,
            "score"       : round(hybrid_score, 4),
            "vector_score": scores["vector"],
            "bm25_score"  : scores["bm25"],
            "page"        : scores["meta"]["page"],
            "source"      : scores["meta"]["source"]
        })

    # ordering
    output = sorted(output, key=lambda x: x["score"], reverse=True)[:top_k]

    print(f"\n Hybrid Search (α={alpha}):")
    for i, r in enumerate(output[:3]):
        print(f"  [{i+1}] Hybrid: {r['score']:.4f} | "
              f"Vec: {r['vector_score']:.4f} | "
              f"BM25: {r['bm25_score']:.4f} | "
              f"Page: {r['page']}")

    return output

# **16. Score Filtering**

In [ ]:
def filter_by_score(results, threshold=0.45):

    filtered = [r for r in results if r["score"] >= threshold]

    print(f"\n  Score Filtering:")
    print(f"    before  : {len(results)} ")
    print(f"    after  : {len(filtered)}   (threshold={threshold})")

    if not filtered:
        print("No relevant results found")

    return filtered

# **17. Cross-Encoder Reranking**

This improves ranking accuracy using a Cross-Encoder model.

In [ ]:
#Initialize Cross-Encoder Reranker
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-Encoder reranker initialized.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder reranker initialized.


## **Reranking Function**

In [ ]:
def rerank_results(query, results, top_n=5):
    if not results:
        print("No results to rerank")
        return []

    # Cross-Encoder  pairs: (query, chunk_text)
    pairs     = [(query, r["text"]) for r in results]
    ce_scores = reranker.predict(pairs)

    for i, r in enumerate(results):
        r["rerank_score"] = round(float(ce_scores[i]), 4)

    reranked = sorted(results, key=lambda x: x["rerank_score"], reverse=True)

    print(f"\n Reranking (Top {top_n}):")
    for i, r in enumerate(reranked[:top_n]):
        print(f"  [{i+1}] Rerank: {r['rerank_score']:.4f} | "
              f" before: {r['score']:.4f} | "
              f"Page: {r['page']}")
        print(f"       {r['text'][:100]}...")

    return reranked[:top_n]

# **18. Enhanced Retrieval Pipeline**

In [ ]:
def enhanced_search(
    query,
    top_k=10,
    threshold=0.45,
    final_top=3,
    alpha=0.6
):

    results = hybrid_search(
        query,
        top_k=top_k,
        alpha=alpha
    )

    results = filter_by_score(
        results,
        threshold=threshold
    )

    results = rerank_results(
        query,
        results,
        top_n=final_top
    )

    return results

In [ ]:
results = enhanced_search(
    query="What is sentiment analysis?",
    top_k=10,
    threshold=0.45,
    final_top=3,
    alpha=0.6
)


 Hybrid Search (α=0.6):
  [1] Hybrid: 0.4628 | Vec: 0.7713 | BM25: 0.0000 | Page: 2
  [2] Hybrid: 0.4592 | Vec: 0.7653 | BM25: 0.0000 | Page: 3
  [3] Hybrid: 0.4586 | Vec: 0.7643 | BM25: 0.0000 | Page: 6

  Score Filtering:
    before  : 10 
    after  : 6   (threshold=0.45)

 Reranking (Top 3):
  [1] Rerank: -10.5110 |  before: 0.4553 | Page: 1
       Language Modeling and N-grams Introduction to Language Modeling Language modeling focuses on predict...
  [2] Rerank: -11.1525 |  before: 0.4501 | Page: 5
       Higher n provides more context but causes data sparsity. N-gram Prediction Uses previous n 1 words t...
  [3] Rerank: -11.1761 |  before: 0.4586 | Page: 6
       Beyond N-grams Neural networks (RNNs, LSTMs) handle longer context. Transformers process entire sequ...


# **19. Prompt Builder**

In [ ]:
def build_prompt(question, retrieved_chunks, max_chunks=3):

    selected_chunks = retrieved_chunks[:max_chunks]

    context = "\n\n".join(
        f"[Doc {i+1}] {chunk.strip()}"
        for i, chunk in enumerate(selected_chunks)
        if chunk and len(chunk.strip()) > 10
    )

    prompt = f"""
You are a STRICT RAG system.

================ RULES ================
- Answer ONLY using the provided context.
- Do NOT use external knowledge.
- If answer exists even partially → use it.
- If NOT found, reply EXACTLY:
  The information is not available in the documents.
- Keep answer short (2–5 lines).
======================================

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
    return prompt

# **20. Openrouter API  Integration**

## **Configure API**

In [ ]:
import requests

API_KEY = "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"

In [ ]:
import requests

def generate_with_openrouter(prompt):

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "meta-llama/llama-3.1-8b-instruct",
        "messages": [
            {
                "role": "system",
                "content": "You are a STRICT RAG assistant. Answer ONLY from provided context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.1
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        data = response.json()

        if response.status_code != 200:
            return f"API Error: {data}"

        if "choices" not in data:
            return f"Bad Response: {data}"

        return data["choices"][0]["message"]["content"]

    except Exception as e:
        return f"Request failed: {e}"

## **Generate Answer Function**

In [ ]:
def generate_answer(query):

    # 1. Retrieve
    retrieved_results = enhanced_search(
        query=query,
        top_k=10,
        threshold=0.45,
        final_top=4
    )

    if not retrieved_results:
        return "The information is not available in the documents."

    # 2. Clean chunks
    chunks_text = [
        r["text"].strip()
        for r in retrieved_results[:2]
        if r.get("text") and len(r["text"]) > 20
    ]

    # 3. Build prompt
    final_prompt = build_prompt(query, chunks_text)

    # 4. Generate
    return generate_with_openrouter(final_prompt)

## **21. Test the System**

In [ ]:
questions = [
    "What is tokenization?",
    "Why remove stopwords?",
    "What are steps of sentiment analysis?",
    "How is sentiment score calculated?"
]

for q in questions:
    print("\n" + "="*60)
    print("Q:", q)
    print("="*60)

    answer = generate_answer(q)
    print(answer)


Q: What is tokenization?

 Hybrid Search (α=0.6):
  [1] Hybrid: 0.4948 | Vec: 0.8246 | BM25: 0.0000 | Page: 3
  [2] Hybrid: 0.4807 | Vec: 0.8011 | BM25: 0.0000 | Page: 16
  [3] Hybrid: 0.4797 | Vec: 0.7995 | BM25: 0.0000 | Page: 7

  Score Filtering:
    before  : 10 
    after  : 9   (threshold=0.45)

 Reranking (Top 4):
  [1] Rerank: 3.4644 |  before: 0.4948 | Page: 3
       Machine translation and speech recognition. Virtual assistants that respond to voice commands. How L...
  [2] Rerank: 2.3165 |  before: 0.4660 | Page: 4
       They can be words, subwords, or characters. Subword tokenization improves efficiency and handles unk...
  [3] Rerank: -0.5191 |  before: 0.4797 | Page: 7
       Summary Language models predict text using probability. N-grams are foundational statistical models....
  [4] Rerank: -0.6676 |  before: 0.4807 | Page: 16
       Step 1: Tokenization deep, learning, models, require, large, datasets Step 2: Generate 4-grams Slidi...
Tokens are the basic units proce

In [ ]:
!pip install gradio

# **22. Gradio Chat Interface**

In [ ]:
import gradio as gr
import traceback

uploaded_chunks = []

def load_pdf(file):
    global uploaded_chunks
    try:
        if file is None:
            return "No file uploaded"

        file_path = file  # بييجي string مباشرة
        uploaded_chunks = process_pdfs([file_path])
        return f"PDF processed successfully | Chunks: {len(uploaded_chunks)}"

    except Exception as e:
        return f"ERROR: {str(e)}\n\n{traceback.format_exc()}"

def rag_retrieval(query):
    global uploaded_chunks
    if not uploaded_chunks:
        return []
    results = enhanced_search(query=query, top_k=10, threshold=0.45, final_top=4)
    return results

def rag_interface(question):
    if not uploaded_chunks:
        return "No documents loaded. Please upload a PDF first."
    results = rag_retrieval(question)
    if not results:
        return "The information is not available in the documents."
    chunks_text = [r["text"] for r in results[:2]]
    final_prompt = build_prompt(question, chunks_text)
    return generate_with_openrouter(final_prompt)

demo = gr.Blocks()
with demo:
    gr.Markdown("# 🔎 RAG System")
    file_input = gr.File(label="Upload PDF", type="filepath")
    upload_btn = gr.Button("Process PDF")
    status = gr.Textbox()
    upload_btn.click(load_pdf, inputs=file_input, outputs=status)
    gr.Markdown("## Ask Questions")
    question = gr.Textbox(label="Question")
    answer = gr.Textbox(label="Answer")
    ask_btn = gr.Button("Generate Answer")
    ask_btn.click(rag_interface, inputs=question, outputs=answer)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f0d0b1b30c5a1aa8cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#  RAG System Project Summary

##  Overview
This project implements a **Retrieval-Augmented Generation (RAG) system** that combines:
- Hybrid Search (Vector + BM25)
- Reranking model
- LLM (OpenRouter - LLaMA 3.1)

---

##  Pipeline Steps

### 1. Query Processing
User question is received from UI (Gradio).

### 2. Hybrid Retrieval
- Vector Search (semantic similarity)
- BM25 Search (keyword matching)
- Combined using weighted score (α = 0.6)

---

### 3. Reranking
- Cross-encoder model reranks top documents
- Keeps most relevant chunks only

---

### 4. Prompt Construction
- Strict RAG prompt is created
- Only retrieved documents are included

---

### 5. LLM Generation
- OpenRouter API (LLaMA 3.1 8B Instruct)
- Temperature = 0.1 (low hallucination)

---

##  Key Features
- Strict context-based answering
- No external knowledge allowed
- Hybrid retrieval system
- Reranking for better accuracy
- Gradio UI for interaction

---

##  Output Behavior
- If answer exists → extracted from documents
- If not found → "The information is not available in the documents."